# SyxEconomyMod — Rebalancing Diagnostic Dashboard

Dieses Notebook liest die CSV-Exporte des `DiagnosticExporter` ein und erzeugt die vier Analyse-Plots:

1. **Resource Scarcity Heatmap** — Welche Ressourcen sind knapp?
2. **Macro Trend Stacked** — Bevölkerung, Gini, Staatskasse, Löhne, Audit-Delta
3. **Anchor vs Market Price Drift** — Pro Resource: wie stark driftet der Marktpreis vom Ankerpreis?
4. **Gini vs Treasury** — Trade-off zwischen Ungleichheit und Staatsvermögen

## Setup

```bash
pip install pandas matplotlib numpy
```

CSV-Dateien müssen zuerst im Spiel exportiert worden sein:
```java
EconConfig.diagnosticsExportEnabled = true
```
Dateien landen in `~/.local/share/songsofsyx/mods/SyxEconomyMod/diagnostics/`.

In [ ]:
import sys
import os

# Falls das Skript in tools/ liegt, Pfad ergänzen
script_dir = os.path.join(os.getcwd(), "tools")
if os.path.isdir(script_dir):
    sys.path.insert(0, script_dir)

import rebalance_plots as rp
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})

print("Module geladen.")

## Daten laden

Passe `DIAG_DIR` an, falls die CSV-Dateien woanders liegen.

In [ ]:
DIAG_DIR = os.path.expanduser("~/.local/share/songsofsyx/mods/SyxEconomyMod/diagnostics")

df_macro, df_res = rp.load_data(DIAG_DIR)

print(f"Macro:  {len(df_macro)} Tage, {len(df_macro.columns)} Spalten")
print(f"Res:    {len(df_res)} Zeilen, {df_res['resource'].nunique()} Ressourcen")
print(f"Tage:   {df_macro['game_day'].min()} – {df_macro['game_day'].max()}")
print(f"\nGini (ø):     {df_macro['gini'].mean():.4f}")
print(f"Treasury (ø):  {df_macro['treasury'].mean():,.0f}")
print(f"Population:    {df_macro['population'].iloc[-1]:,}")

---
## Plot 1: Resource Scarcity Heatmap

Jede Zeile = eine Resource. Farbe = `days_of_supply` (wie viele Tage reicht der Vorrat bei aktueller Nachfrage).
- **Rot** (< 3 Tage) = akuter Engpass
- **Grün** (> 15 Tage) = Überschuss
- **Weiß** = kein Bedarf (Ressource wird aktuell nicht gehandelt)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 10))
rp.plot_scarcity_heatmap(df_res, ax=ax, max_resources=50)
fig.tight_layout()
plt.show()

---
## Plot 2: Macro Trend Stacked

Sechs Subplots mit den wichtigsten Makro-Indikatoren im Zeitverlauf.

In [ ]:
fig = rp.plot_macro_trends(df_macro, figsize=(18, 16))
plt.show()

---
## Plot 3: Anchor vs Market Price Drift

Für die 9 Ressourcen mit der stärksten Abweichung zwischen Ankerpreis und Marktpreis:
- **Blau** = Anchor-Preis (zentral gesetzt)
- **Orange** = Market-Preis (tatsächlich erzielt)
- **Grün (rechte Achse)** = Ratio Market/Anchor → 1.0 = perfekte Übereinstimmung

In [ ]:
fig = rp.plot_price_drift(df_res, top_n=9, figsize=(18, 14))
plt.show()

---
## Plot 4: Gini vs Treasury (Dual-Axis)

Gini (rot, linke Achse, 0–1) vs Treasury (grün, rechte Achse).
Optionale Zusatz-Linien: Food Days (gestrichelt) und Mean Wage (Strich-Punkt).
Saisonale Hintergrundbänder markieren Frühling/Sommer/Herbst/Winter.

In [ ]:
fig = rp.plot_gini_treasury(
    df_macro,
    figsize=(16, 7),
    include_food=True,
    include_wages=True,
)
plt.show()

---
## Zusätzliche Exploration

### Top-5 knappste Ressourcen (nach ∅ days_of_supply)

In [ ]:
scarcity_ranking = (
    df_res[df_res["days_of_supply_clean"].notna()]
    .groupby("resource")["days_of_supply_clean"]
    .mean()
    .sort_values()
    .head(10)
)
print("Top-10 knappste Ressourcen (Ø days_of_supply):")
display(scarcity_ranking.to_frame(name="days_of_supply"))

### Preis-Drift Ranking

In [ ]:
drift_ranking = (
    df_res.groupby("resource")
    .apply(lambda g: (g["price_ratio"] - 1.0).abs().mean(), include_groups=False)
    .sort_values(ascending=False)
    .head(10)
)
print("Top-10 Ressourcen nach Preis-Drift (|ratio − 1|):")
display(drift_ranking.to_frame(name="drift_magnitude"))

### Starving-Signal-Trend

In [ ]:
starving_by_day = df_res.groupby("game_day")["starving_signal"].sum()
ax = starving_by_day.plot(
    figsize=(14, 3),
    title="Starving Resources per Day (days_of_supply < 3)",
    color="#d62728",
    alpha=0.7,
    linewidth=0.8,
)
ax.set_ylabel("# Resources with starving signal")
ax.grid(True, alpha=0.3)
plt.show()